# Capstone 3: Art Gallery Search Pipeline

This is the least guided notebook in the programme. You get a brief,
some data, and a target output. How you get there is your decision.

The skills involved -- text cleaning, chunking, vector search, RAG
prompt construction -- are among the most in-demand in the industry
right now. But they are also among the most misunderstood. Everyone
wants to build RAG. Few people think carefully about the data
engineering underneath it. That is what this capstone is about.

## The brief

The Whitmore Gallery is a contemporary art gallery in East London.
They have a growing collection of exhibition catalogues and artist
biographies -- rich, detailed texts about their exhibitions and the
artists they represent.

They want visitors and researchers to be able to search this
collection using natural language. Not keyword search. Semantic
search. Something like:

> "Show me abstract expressionist works influenced by the natural world."

> "Which artists work with industrial materials?"

> "Tell me about photography exhibitions that deal with environmental themes."

Your job: build the data pipeline that makes this possible.

You are not building a production system. You are building a
proof-of-concept that demonstrates the approach and produces
credible results. The gallery's tech team will take it from there.

## The data

In `../data/` you will find:

- **`gallery_catalogues.json`** -- 30 entries: exhibition catalogues
  and artist biographies. Each entry has an `id`, `type` (exhibition
  or artist_biography), `title`, `year` (for exhibitions), and `text`
  (the full catalogue text, 300-800 words each).

- **`gallery_embeddings.npy`** -- Pre-computed TF-IDF embeddings for
  text chunks derived from the catalogue texts. These are not
  transformer embeddings (we cannot run those in the browser), but
  TF-IDF vectors capture term importance and allow meaningful
  similarity search.

- **`gallery_chunks.json`** -- The text chunks that correspond to the
  embeddings (in the same order).

- **`gallery_chunk_metadata.json`** -- Metadata for each chunk:
  source_id, source_title, and chunk_index.

## Setup

In [ ]:
%pip install -q scikit-learn matplotlib

In [ ]:
import json
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

---

## Step 1: Load and explore the data

Start by loading the gallery catalogues and getting a feel for the
content. Read some of the texts. Understand the kinds of topics,
names, and terminology that appear. This matters -- you cannot build
a good search pipeline for data you have not read.

### Task: Load and explore

In [ ]:
# Your code here -- load gallery_catalogues.json and explore


---

## Step 2: Clean the texts

Build a text cleaning pipeline for the gallery texts. Think about
what processing is appropriate for this domain. Some things to
consider:

- Should you lowercase everything? (Careful -- artist names and
  exhibition titles are proper nouns.)
- Are there special characters, unusual whitespace, or encoding
  artefacts to handle?
- Should you remove stopwords? (Think about what matters for
  semantic search vs keyword search.)
- What about normalising punctuation -- curly quotes, em dashes,
  that sort of thing?

The right answer depends on what you are optimising for. A search
pipeline needs to preserve enough meaning for queries like
"abstract expressionist works influenced by nature" to find
relevant results. Aggressive cleaning that strips away meaningful
words will hurt retrieval quality.

Write a `clean_text()` function and apply it to all catalogue texts.

### Task: Build a text cleaning function

In [ ]:
def clean_text(text):
    """Clean a gallery catalogue text for search indexing.

    Consider:
    - Normalising whitespace
    - Handling special characters
    - Preserving meaningful terms (artist names, art movements)
    - Keeping case for proper nouns (or lowercasing -- your choice)

    Returns:
        Cleaned text string.
    """
    # Your code here
    pass

### Task: Apply cleaning to all texts

In [ ]:
# Your code here


---

## Step 3: Chunk the texts

Full catalogue texts are too long to use as search units. A query
like "artists who work with industrial materials" should match a
specific passage, not an entire 800-word biography.

You need a chunking strategy. Options include:

- **Fixed-size word chunks** with overlap (e.g. 200 words, 40-word
  overlap). Simple and predictable.
- **Sentence-based chunks** grouped into blocks. Better preserves
  semantic boundaries.
- **Paragraph-based chunks.** Respects the author's structure but
  chunk sizes vary wildly.

There is no single right answer. The trade-offs are real: smaller
chunks give more precise retrieval but lose context. Larger chunks
give more context but reduce precision. Overlap helps ensure that
important information near chunk boundaries is not lost.

Choose a strategy, implement it, and be prepared to justify your
choice.

### Task: Implement your chunking strategy

In [ ]:
def chunk_text(text, source_id, source_title):
    """Split text into chunks for search indexing.

    Args:
        text: The cleaned text to chunk.
        source_id: ID of the source document.
        source_title: Title of the source document.

    Returns:
        List of dicts, each with 'text', 'source_id', 'source_title',
        and 'chunk_index'.
    """
    # Your code here
    pass

### Task: Chunk all documents and inspect the results

In [ ]:
# Your code here -- chunk all documents and report statistics
# How many chunks? What is the distribution of chunk sizes?


---

## Step 4: Build the search index

We have two options here:

**Option A:** Use the pre-computed embeddings from
`gallery_embeddings.npy` and `gallery_chunks.json`. These were
generated by fitting a `TfidfVectorizer(max_features=384)` on
the original chunks. To search, you would fit the same vectorizer
on the chunks and use it to transform query strings.

**Option B:** Build your own embeddings from scratch using your
cleaned and chunked texts. This gives you full control and means
your embeddings match your cleaning and chunking decisions.

Option B is better practice. If you cleaned or chunked differently
from the pre-computed version, the pre-computed embeddings will not
align with your chunks anyway. We recommend building your own.

The process:
1. Fit a `TfidfVectorizer` on all your chunks
2. Transform all chunks to get the embedding matrix
3. For a search query, transform it with the same vectorizer
4. Compute cosine similarity between the query vector and all
   chunk vectors
5. Return the top-k most similar chunks

### Task: Build the search index

In [ ]:
# Your code here -- fit TfidfVectorizer and create embedding matrix


### Task: Implement the search function

In [ ]:
def search(query, top_k=5):
    """Search the gallery catalogue using semantic similarity.

    Args:
        query: A natural language search query.
        top_k: Number of results to return.

    Returns:
        List of dicts with 'text', 'source_title', 'score', etc.
    """
    # Your code here
    pass

### Task: Test with sample queries

Try these queries and assess the quality of the results:

1. "abstract expressionist works influenced by the natural world"
2. "artists who use industrial materials"
3. "photography exhibitions about environmental damage"
4. "digital art and artificial intelligence"
5. "sculpture installations that use light"

For each query, look at the top 3 results. Are they relevant?
Are the right documents being retrieved? Where does the search
fall short?

In [ ]:
# Your code here -- test queries and print results


---

## Step 5: RAG prompt construction

Retrieval-Augmented Generation means: retrieve relevant context,
then construct a prompt that gives an LLM both the question and
the retrieved information. The LLM generates an answer grounded
in actual data rather than its training knowledge alone.

We cannot call an LLM from within Pyodide. But we can build the
prompt that you would send to one. This is genuinely important
work -- prompt construction is data engineering, not magic.

A good RAG prompt typically includes:

1. **System instruction:** What role the LLM should take, how it
   should behave, what it should not do (e.g. "only answer based
   on the provided context")
2. **Retrieved context:** The top-k chunks from your search,
   clearly labelled with their source
3. **The user's question**
4. **Output format guidance:** How the answer should be structured

### Task: Build a RAG prompt constructor

In [ ]:
def build_rag_prompt(query, search_results):
    """Construct a RAG prompt from a query and retrieved chunks.

    Args:
        query: The user's question.
        search_results: List of dicts from the search function.

    Returns:
        A string containing the complete prompt, ready to send
        to an LLM.
    """
    # Your code here
    pass

### Task: Generate example prompts

Use your search and prompt functions together. For two or three
queries, print the complete prompt. Read it critically.

- Does the context contain relevant information?
- Is there enough context for an LLM to give a useful answer?
- Is the instruction clear about what the LLM should and should
  not do?
- Would the prompt work well with a real LLM?

In [ ]:
# Your code here


---

## Step 6: Evaluate search quality

A search system is only useful if it returns relevant results.
We need some way to measure quality, even without user feedback.

Here is a simple evaluation approach:

1. Write 5-8 test queries with known relevant documents. For
   example, a query about "Rachel Lim" should retrieve her
   biography and the exhibitions she appears in.

2. For each query, check whether the expected documents appear
   in the top-k results.

3. Calculate precision@k: of the top-k results, what fraction
   are relevant?

This is a manual evaluation, which is fine for a proof-of-concept.
In production you would build a proper evaluation set with human
judgements.

### Task: Create test queries with expected results

In [ ]:
# Define your test queries and expected relevant source IDs
test_queries = [
    {
        "query": "Rachel Lim sculptures using light",
        "expected_sources": ["ART-003", "EXH-003"],
    },
    # Your additional test queries here
]

### Task: Run evaluation and calculate precision

In [ ]:
# Your code here -- run search for each test query, check results,
# calculate precision@k


---

## Step 7: The production gap

This is the most important section of the notebook, and it
contains no code at all.

What we have built works. It retrieves relevant text, it constructs
reasonable prompts, and for a proof-of-concept it is solid. But it
is also, honestly, a long way from production. Understanding that
gap -- clearly, specifically, without either dismissing the work or
overselling it -- is what separates a junior engineer from a
senior one.

### What we used vs what production needs

| What we did | What production requires |
|-------------|-------------------------|
| TF-IDF vectors | Transformer embeddings (e.g. from OpenAI, Cohere, or an open-source model like `all-MiniLM-L6-v2`) |
| NumPy array in memory | A vector database (Pinecone, Weaviate, Qdrant, pgvector) |
| Static data | Incremental indexing as new catalogues are published |
| No caching | Query result caching for common searches |
| Manual evaluation | Automated relevance testing, user feedback loops, A/B testing |
| No monitoring | Latency tracking, retrieval quality metrics, cost monitoring |
| Print the prompt | Actually call an LLM and return generated answers |

### Your turn

**Answer these questions. Be specific, not vague.**

**1. Why are transformer embeddings better than TF-IDF for this
use case?**

Think about: synonyms, paraphrases, semantic understanding vs
term matching. What queries would TF-IDF fail on that transformers
would handle?

*Your answer:*

...

**2. What are the risks of deploying this as a public-facing tool?**

Think about:
- Hallucination: the LLM might generate plausible but false
  information about artworks or artists
- Bias: the LLM might reproduce biases present in its training
  data when discussing art and artists
- Copyright: are there legal concerns about indexing and
  reproducing catalogue text?
- Prompt injection: could a malicious query cause the LLM to
  behave unexpectedly?

*Your answer:*

...

**3. How would you measure success in production?**

What metrics would you track? How would you know if the search
is actually helping visitors and researchers?

*Your answer:*

...

**4. What would the data pipeline look like at scale?**

The gallery adds 4-6 new exhibitions per year. Each generates
catalogue text, press releases, artist statements, and reviews.
How do you ingest, process, and index this new content? What
about updating embeddings when the model changes? What about
versioning?

*Your answer:*

...

**5. What would you present to the gallery's director?**

You have 15 minutes with a non-technical stakeholder. They want
to know: does this work, what does it cost, what are the risks,
and should we invest further? What do you show them? What do
you tell them honestly?

*Your answer:*

...

---

## Summary

You have built a text processing and retrieval pipeline from
scratch: cleaning, chunking, embedding, searching, and prompt
construction. More importantly, you have thought critically about
what this pipeline can and cannot do, and what production
deployment would actually require.

That honesty about the gap between proof-of-concept and production
is something employers value enormously. Anyone can follow a RAG
tutorial. The people who get hired are the ones who can explain
what breaks when you move from a Jupyter notebook to a real
system serving real users.

This is the end of the capstone projects and the end of the
taught programme. Everything you have learned -- Python, SQL,
data modelling, ETL, data quality, DuckDB, text processing,
vector search -- these are the tools. What matters now is
what you build with them.